# Simulated spatial domains: recover the five reference regions

The simulation contains 1,296 observations with RNA, ADT and ATAC. Four spatial factors and a zero-factor background define five reference classes.

Training is commented out by default. **Run All** uses saved CellSTIC results and recomputes the analyses below. Figures and tables are saved alongside the existing model results under `DATA / "cellstic"`. Run All does not train or modify the saved model.

## 1. Set up the dataset

Start Jupyter from this repository. Use the environment in `environment.yml` and supply the assembled inputs and saved results at the paths below.

In [1]:
from pathlib import Path
import sys
import json
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline/cellstic.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline.analysis import Analysis
from utils.analysis.result_io import load_result, lr_matrices, channel_keys
from utils.analysis import dataset_analysis as da
%matplotlib inline
plt.style.use("default")
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

/home/wangshuai/anaconda3/envs/spaGem/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA = ROOT / "paper/data/domain_sim"
RESULT_PATH = DATA / "cellstic/out/out.h5ad"
OUTPUT = DATA / "cellstic"
OUTPUT.mkdir(parents=True, exist_ok=True)

## 2. Train CellSTIC (optional)

Uncomment both cells to fit the assembled inputs. The pipeline performs normalization, feature extraction, graph construction and training. If explicitly enabled, training writes to the same output location used below and replaces that saved fit. Only feature integration and spatial clustering are fitted for this task.

In [3]:
# from model.train import get_cellstic_config
# from pipeline.cellstic import run_cellstic
# names = ['rna', 'adt', 'atac']
# inputs = [ad.read_h5ad(DATA / "h5ad" / f"{name}.h5ad") for name in names]
# config = get_cellstic_config('domain_sim')

In [4]:
# trained = run_cellstic(inputs, config, modality_types=["rna", "adt", "atac"],
#                       auto_n_clusters=5, run_ccc=False, output_path=OUTPUT)
# RESULT_PATH = OUTPUT / "out/out.h5ad"

## 3. Load the saved result

Load the spatial metadata, embedding and predicted graphs. Expression and auxiliary calibration arrays stay on disk.

In [5]:
result = load_result(RESULT_PATH)
display(pd.DataFrame({"observations": [result.n_obs],
                      "domains": [result.obs["cluster"].nunique()],
                      "LR channels": [len(channel_keys(result.obsp, result.uns["cellstic"]))]}))

,observations,domains,LR channels
0,1296,5,0


## 4. Inspect spatial domain recovery

The largest spatial factor defines the reference class; all-zero factors define
the background. ARI and NMI are shown with the learned-domain map and compare
the partitions without matching arbitrary cluster numbers. Colors in the two
maps are assigned independently. Reference labels are used only after fitting.

In [6]:
from utils.metrics.spatial_viz import SpatialVisualizer
from utils.analysis.figure_export import show_image_row
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
factors = np.asarray(result.obsm["spfac"])
labels = np.argmax(factors, axis=1).astype(str)
labels[np.all(factors == 0, axis=1)] = "background"
result.obs["spatial_truth"] = pd.Categorical(labels)
ari = adjusted_rand_score(labels, result.obs["cluster"])
nmi = normalized_mutual_info_score(labels, result.obs["cluster"])
domain_dir = OUTPUT / "domain"
domain_dir.mkdir(parents=True, exist_ok=True)
for label, filename in [("spatial_truth", "spatial_truth.svg"), ("cluster", "spatial_domain.svg")]:
    spatial = ad.AnnData(obs=result.obs.copy(), obsm={"spatial": result.obsm["spatial"]})
    spatial.obs["cluster"] = pd.Categorical(result.obs[label].astype(str))
    SpatialVisualizer.generate_spatial_domain_visualization(spatial, save_path=domain_dir / filename)
show_image_row([domain_dir / "spatial_truth.svg", domain_dir / "spatial_domain.svg"],
               labels=["Ground truth", f"Learned domains · ARI = {ari:.3f} · NMI = {nmi:.3f}"])


/home/wangshuai/anaconda3/envs/spaGem/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Visualizing 1296 points with 5 domains in grid style
  Calculated min nearest neighbor distance: 0.1143
  Square size (base * 1.0): 0.1371
  Direct visualization: 1296 points, each point gets its own square


Spatial domain visualization (grid style) saved to /home/wangshuai/project/CellSTIC-dup/paper/data/domain_sim/cellstic/domain/spatial_truth.svg
Visualizing 1296 points with 5 domains in grid style
  Calculated min nearest neighbor distance: 0.1143
  Square size (base * 1.0): 0.1371
  Direct visualization: 1296 points, each point gets its own square


/home/wangshuai/anaconda3/envs/spaGem/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Spatial domain visualization (grid style) saved to /home/wangshuai/project/CellSTIC-dup/paper/data/domain_sim/cellstic/domain/spatial_domain.svg
